[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/reranking.ipynb)

# Chapter 3: Reranking Retrieved Results

This notebook demonstrates how to use cross-encoder reranking models (BGE Reranker) to improve the relevance ordering of retrieved documents.

**What you'll learn:**
- Load and use the BGE Reranker v2 cross-encoder model
- Score query-document pairs for relevance
- Rerank retrieved documents by relevance score

**Prerequisites:** `pip install sentence-transformers`

## Setup

We install the latest sentence-transformers and import the CrossEncoder class, which scores query-document pairs directly rather than comparing embeddings.

In [1]:
!pip install -U sentence-transformers --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from sentence_transformers.cross_encoder import CrossEncoder

## Prepare Query and Documents

We define a query and a mixed-relevance document set that simulates a typical retriever output — some passages are on-topic, some are tangential, and some are irrelevant.

In [3]:
query = "What is the main benefit of using a transformer model in NLP?"

documents = [
    "Recurrent Neural Networks (RNNs) were previously popular for sequence tasks.", # Less relevant
    "Transformers allow for parallel processing of input tokens, leading to faster training times compared to RNNs.", # Highly relevant (speed/parallelism)
    "BERT, a popular transformer model, achieves state-of-the-art results on many NLP benchmarks.", # Relevant context, but not the *main benefit* of the architecture itself
    "The attention mechanism in transformers enables the model to weigh the importance of different words in the input sequence.", # Highly relevant (attention mechanism)
    "Convolutional Neural Networks (CNNs) are primarily used in computer vision.", # Irrelevant
    "A key advantage of the transformer architecture is its ability to handle long-range dependencies more effectively than RNNs.", # Highly relevant (long-range dependencies)
    "You can fine-tune pre-trained transformer models for specific downstream tasks." # Related benefit, but maybe not the *main* architectural one
]

In [4]:
# You can find other BGE models on Hugging Face: https://huggingface.co/BAAI
model_name = 'BAAI/bge-reranker-v2-m3'

# The model will be downloaded automatically the first time you run this.
# You can specify the device (e.g., 'cuda', 'cpu') if needed.
# device='cuda' if torch.cuda.is_available() else 'cpu' # Requires PyTorch installed
try:
    # Let sentence-transformers handle device placement automatically
    # It usually defaults to GPU if available and compatible, otherwise CPU.
    model = CrossEncoder(model_name)
    print(f"Successfully loaded BGE Reranker model: {model_name}")
    print("-" * 30)
except Exception as e:
    print(f"Error loading model '{model_name}': {e}")
    print("Please ensure 'sentence-transformers' is installed (`pip install -U sentence-transformers`)")
    print("Also ensure you have PyTorch or TensorFlow installed.")
    exit()

Successfully loaded BGE Reranker model: BAAI/bge-reranker-v2-m3
------------------------------


## Score and Rerank

The cross-encoder evaluates each query-document pair directly, producing a relevance score that lets us re-order results by true semantic match rather than embedding distance.

In [5]:
# 1. Prepare input for the CrossEncoder
# The model expects a list of pairs: [query, document].
sentence_pairs = [[query, doc] for doc in documents]

# 2. Compute scores
# The predict method calculates the relevance score for each pair.
# Higher scores indicate higher relevance between the query and the document.
# The 'show_progress_bar=True' is helpful for larger lists of documents.
print("Calculating relevance scores...")
scores = model.predict(sentence_pairs, show_progress_bar=True)

print("-" * 30)
print("Raw Scores:", scores)
print("-" * 30)

# 3. Rank documents based on scores
# Combine documents with their scores and sort in descending order of score.

docs_with_scores = list(zip(documents, scores))
reranked_docs = sorted(docs_with_scores, key=lambda x: x[1], reverse=True)

# 4. Display the results

print("Query:", query)
print("\n--- Original Document Order ---")
for i, doc in enumerate(documents):
    print(f"{i+1}. {doc}")

print("\n--- Reranked Document Order ---")
print("(Higher score indicates higher relevance)")
for i, (doc, score) in enumerate(reranked_docs):
    print(f"{i+1}. Score: {score:.4f} - {doc}")

Calculating relevance scores...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

------------------------------
Raw Scores: [4.0928167e-05 2.1364099e-01 5.9133494e-01 2.1382652e-01 8.6146996e-05
 8.3851719e-01 2.4680912e-02]
------------------------------
Query: What is the main benefit of using a transformer model in NLP?

--- Original Document Order ---
1. Recurrent Neural Networks (RNNs) were previously popular for sequence tasks.
2. Transformers allow for parallel processing of input tokens, leading to faster training times compared to RNNs.
3. BERT, a popular transformer model, achieves state-of-the-art results on many NLP benchmarks.
4. The attention mechanism in transformers enables the model to weigh the importance of different words in the input sequence.
5. Convolutional Neural Networks (CNNs) are primarily used in computer vision.
6. A key advantage of the transformer architecture is its ability to handle long-range dependencies more effectively than RNNs.
7. You can fine-tune pre-trained transformer models for specific downstream tasks.

--- Reranked Do

> **Note:** The cross-encoder reranker scores each document independently against the query (0 = irrelevant, 1 = perfectly relevant). Compare the original order with the reranked order: the passage about "long-range dependencies" jumped from position 6 to position 1 (score 0.84), while irrelevant passages like "CNNs are used in computer vision" dropped to near-zero. This is why reranking is a key stage in production RAG pipelines — initial retrieval casts a wide net, then reranking sharpens relevance.